In [5]:
from google.colab import drive
import os, json, re, csv
from collections import Counter, defaultdict

drive.mount('/content/drive')

# ---- set these to your file on THIS account's Drive ----
SC_PATH    = '/content/drive/MyDrive/third_try/baseline__private__samples.jsonl'   # <- your file
OUT_CSV    = '/content/drive/MyDrive/third_try/submission.csv'           # <- output
ID_FIELD   = 'id'
TEXT_FIELD = 'samples'

def find_boxed(s):
    out, i = [], 0
    while True:
        j = s.find(r'\boxed{', i)
        if j < 0: break
        k = j + 7; depth = 1; start = k
        while k < len(s) and depth:
            depth += (s[k] == '{') - (s[k] == '}'); k += 1
        out.append(s[start:k-1]); i = k
    return out

def extract_answer(text):
    seg = text.rsplit('</think>', 1)[-1] if '</think>' in text else text
    boxes = find_boxed(seg) or find_boxed(text)
    return ", ".join(boxes) if boxes else None

def norm(a):
    if a is None: return None
    for t in (' ', '$', '\\left', '\\right', '\\!', '\\,'):
        a = a.replace(t, '')
    return a.strip().lower()

def get_text(x):                         # a sample may be a string or a dict
    if isinstance(x, str): return x
    if isinstance(x, dict):
        strs = [v for v in x.values() if isinstance(v, str)]
        return max(strs, key=len) if strs else ""   # the trace is the longest string
    return str(x)

by_id = defaultdict(list)
for line in open(SC_PATH):
    r = json.loads(line)
    for s in r[TEXT_FIELD]:              # the k generations for this question
        resp = get_text(s)
        by_id[r[ID_FIELD]].append((resp, norm(extract_answer(resp))))

final = {}
for rid, lst in by_id.items():
    votes = Counter(n for _, n in lst if n is not None)
    if votes:
        winner = votes.most_common(1)[0][0]
        final[rid] = next(resp for resp, n in lst if n == winner)  # a real trace giving the winner
    else:
        final[rid] = lst[0][0]           # nothing extractable; fall back to first sample

os.makedirs(os.path.dirname(OUT_CSV) or '.', exist_ok=True)   # don't die on a missing dir
with open(OUT_CSV, 'w', newline='') as f:
    w = csv.writer(f, quoting=csv.QUOTE_ALL)
    w.writerow(['id', 'response'])
    for rid in sorted(final):
        w.writerow([rid, final[rid]])

print(f'{len(final)} questions aggregated -> {OUT_CSV}')
ex = next(iter(by_id.values()))
print('sanity (one question vote spread):', Counter(n for _, n in ex if n))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
943 questions aggregated -> /content/drive/MyDrive/third_try/submission.csv
sanity (one question vote spread): Counter({'4,16,4,16': 6, '4,16': 1})
